<a href="https://colab.research.google.com/github/Daudi-Zacharia/msc-thesis/blob/main/classification/clean_notebooks/d2_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# D2 (4200-image) Classification Experiments

Clean rebuild of the **D2 experiments reported in Table IV**: five architectures
across four training schemes.

| Scheme | What changes |
|---|---|
| **No Augmentation** | resize and normalize only |
| **With Augmentation** | flips, small affine jitter and colour jitter added |
| **Oversampling** | no augmentation, but a `WeightedRandomSampler` rebalances the 700 TB / 3500 Normal split |
| **Segmented** | the X-ray multiplied by its predicted lung mask (hard attention), using `seg_unet_2.pth` |

Rebuilt from the per-model notebooks in `classification/raw_notebooks/`. Scratch
and superseded runs are deliberately excluded.

**Checkpoints live in Google Drive, not in this repository:**

```
/content/drive/MyDrive/MSc Data Science/Thesis/Codes/classification
/content/drive/MyDrive/MSc Data Science/Thesis/Codes/segmentation      (VGG16 segmented, seg_unet_2.pth)
```

| Model | No Aug | Aug | Oversampling | Segmented |
|---|---|---|---|---|
| Custom CNN | `class_ex3_run1_4200.pt` | `class_ex3_run2_4200.pt` | `class_ex3_run4_4200.pt` | `class_seg_3cnn_4200.pt` |
| DenseNet169 | `densenet169_rahman_r2_redo.pt` | `densenet169_rahman_r4.pt` | `densenet169_rahman_r5.pt` | `densenet169_class_seg_rahman.pt` |
| VGG16 | `vgg16_rahman_r1_redo_2.pt` | `vgg16_rahman_r2.pt` | `vgg16_rahman_r3_2.pt` | `vgg16_class_seg_4200.pt` |
| ResNet50 | `resnet50_rahman_r3.pt` | `resnet50_rahman_r4.pt` | `resnet50_rahman_r5.pt` | `resnet50_class_seg_4200.pt` |
| EfficientNetB3 | `efficient_net_4200_redo_2.pt` | `efficient_net_4200_run2.pt` | `efficient_net_4200_run3.pt` | `enet_class_seg_4200.pt` |

**Loading conventions differ.** The three unsegmented Custom CNN checkpoints are
state dicts and load into a fresh `CNN()`; every other checkpoint, including the
segmented Custom CNN, is a full model object.

Requires a GPU runtime with Drive mounted. Training cells are kept for
reproducibility but are not meant to be re-run every pass — each Evaluation cell
loads the saved checkpoint instead.

# 1. Setup

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install kagglehub -q
!pip install openpyxl -q
!pip install grad-cam -q

In [ ]:
import os
import random
import shutil
import time

import numpy as np
import pandas as pd
import PIL
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
from torchvision.datasets import ImageFolder

from sklearn.metrics import (confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.manifold import TSNE

# reproducibility
SEED = 7
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# --- Drive locations (checkpoints are NOT in this repo) --------------------
path_classification = '/content/drive/MyDrive/MSc Data Science/Thesis/Codes/classification'
path_segmentation = '/content/drive/MyDrive/MSc Data Science/Thesis/Codes/segmentation'
SEG_CHECKPOINT = os.path.join(path_segmentation, 'seg_unet_2.pth')

# --- local data directories built by this notebook -------------------------
DIR_ORIGINAL = './data_4200'        # 80/10/10 split of the raw X-rays
DIR_FLAT = './Images_4200'          # all 4200 images, unsplit (input to the U-Net)
DIR_ROI = './ROI_Images_4200'       # masked images produced by the U-Net
DIR_SEGMENTED = './data_4200_seg'   # 70/20/10 split of the masked images

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# 2. Helper Functions

Defined once and reused by every model and scheme below. Identical to the
helpers in `d1_experiments.ipynb`.

In [ ]:
def train_model(model, train_dl, val_dl, optimizer, loss_fn, save_path,
                epochs=30, patience=10, save_mode='full', device='cpu'):
    """Trains with early stopping, checkpointing the best validation loss.

    save_mode must match how the corresponding checkpoint was originally stored,
    so that the Evaluation cell can load back what Training wrote:
      'full'       -> torch.save(model, ...)              (the default here)
      'state_dict' -> torch.save(model.state_dict(), ...)
    """
    if save_mode not in ('full', 'state_dict'):
        raise ValueError("save_mode must be 'full' or 'state_dict'")

    model.to(device)

    history = {'Training': {'Loss': [], 'Accuracy': [], 'Time': []},
               'Validation': {'Loss': [], 'Accuracy': [], 'Time': []}}

    best_val_loss = np.inf
    count = 0

    for epoch in range(1, epochs + 1):

        # --- train ---
        model.train()
        t0 = time.time()
        train_loss = 0
        correct = 0

        for batch, target in train_dl:
            inputs, target = batch.to(device), target.to(device)
            output = model(inputs)
            loss = loss_fn(output, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            correct += (output.argmax(dim=1) == target).sum().item()

        history['Training']['Loss'].append(train_loss / len(train_dl))
        history['Training']['Accuracy'].append(correct / len(train_dl.dataset))
        history['Training']['Time'].append((time.time() - t0) / 60)

        # --- validate ---
        model.eval()
        t0 = time.time()
        val_loss = 0
        correct = 0

        with torch.no_grad():
            for batch, target in val_dl:
                inputs, target = batch.to(device), target.to(device)
                output = model(inputs)

                val_loss += loss_fn(output, target).item()
                correct += (output.argmax(dim=1) == target).sum().item()

        val_loss /= len(val_dl)
        val_acc = correct / len(val_dl.dataset)

        history['Validation']['Loss'].append(val_loss)
        history['Validation']['Accuracy'].append(val_acc)
        history['Validation']['Time'].append((time.time() - t0) / 60)

        # --- checkpoint + early stopping ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            count = 0
            torch.save(model.state_dict() if save_mode == 'state_dict' else model,
                       save_path)
            print(f'Best model saved at epoch {epoch}')
        else:
            count += 1

        if count >= patience:
            print(f'Early stopping triggered at epoch {epoch}')
            break

        print(f"Epoch [{epoch}/{epochs}]  "
              f"train loss {history['Training']['Loss'][-1]:.4f}  "
              f"acc {history['Training']['Accuracy'][-1]:.4f}  |  "
              f"val loss {val_loss:.4f}  acc {val_acc:.4f}")

    print(f"\nTotal training time: {sum(history['Training']['Time']):.2f} min")
    return history


def plot_history(history):
    """Training vs validation loss and accuracy."""
    x = range(1, len(history['Training']['Loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    for ax, metric in zip(axes, ['Loss', 'Accuracy']):
        ax.plot(x, history['Training'][metric], label=f'Training {metric}')
        ax.plot(x, history['Validation'][metric], label=f'Validation {metric}')
        ax.set_xlabel('Epochs')
        ax.set_ylabel(metric)
        ax.set_title(f'Training and Validation {metric}')
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
def load_checkpoint(path, how='full', builder=None, device='cpu'):
    """Loads a checkpoint saved either as a full object or as a state dict.

    how='full'        -> torch.load(..., weights_only=False)
    how='state_dict'  -> builder() then load_state_dict
    """
    if how == 'state_dict':
        if builder is None:
            raise ValueError("a builder is required to load a state dict")
        model = builder().to(device)
        model.load_state_dict(torch.load(path, map_location=device))
    else:
        model = torch.load(path, map_location=device, weights_only=False)

    model.to(device)
    model.eval()
    return model


def evaluate_model(model, test_dl, title='', device='cpu'):
    """Runs inference and reports the full metric set, a confusion matrix and a ROC curve.

    zero_division=0 keeps an ill-defined metric explicit (e.g. a model that
    collapses onto one class) instead of raising a warning.
    """
    model.to(device)
    model.eval()

    preds, targets, probs = [], [], []

    with torch.no_grad():
        for batch, target in test_dl:
            output = model(batch.to(device))
            preds.append(output.argmax(dim=1).cpu())
            probs.append(F.softmax(output, dim=1)[:, 1].cpu())
            targets.append(target)

    y_pred = torch.cat(preds).numpy()
    y_true = torch.cat(targets).numpy()
    y_prob = torch.cat(probs).numpy()

    metrics = {
        'Acc.': (y_pred == y_true).mean(),
        'Prec.': precision_score(y_true, y_pred, zero_division=0),
        'Sens.': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'Spec.': recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        'AUC': roc_auc_score(y_true, y_prob),
    }

    print(f'--- {title} ---')
    print(f'Test set: n = {len(y_true)} | TB = {int((y_true == 1).sum())} | '
          f'Normal = {int((y_true == 0).sum())}')
    print('  '.join(f'{k} {v:.4f}' for k, v in metrics.items()))

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fpr, tpr, _ = roc_curve(y_true, y_prob)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Normal', 'TB'], yticklabels=['Normal', 'TB'],
                annot_kws={'size': 16, 'weight': 'bold'}, ax=axes[0])
    axes[0].set_xlabel('Predicted Class')
    axes[0].set_ylabel('True Class')
    axes[0].set_title(f'Confusion Matrix - {title}')

    axes[1].plot(fpr, tpr, color='darkorange', lw=2,
                 label=f"ROC curve (AUC = {metrics['AUC']:.4f})")
    axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[1].set_xlabel('False Positive Rate (1 - Specificity)')
    axes[1].set_ylabel('True Positive Rate (Sensitivity)')
    axes[1].set_title(f'ROC - {title}')
    axes[1].legend(loc='lower right')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    return metrics

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


def find_last_conv_layer(model):
    """Last Conv2d in the network - the usual Grad-CAM target across all five
    architectures used here."""
    last_conv = None
    for _, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            last_conv = module
    if last_conv is None:
        raise ValueError('No convolutional layer found in the model.')
    return last_conv


def plot_gradcam(model, data_dir, transform, mean, std, target_class=1,
                 n_per_class=2, title='', device='cpu'):
    """Grad-CAM overlays for the first n images of each class.

    target_class selects which logit is explained (1 = TB, 0 = Normal); paths are
    sorted so the same images are shown on every run.
    """
    model.to(device)
    # gradients must flow even though the backbone was frozen for training
    for param in model.parameters():
        param.requires_grad = True
    model.eval()

    dataset = ImageFolder(root=data_dir, transform=None)
    class_names = dataset.classes

    selected = []
    for cls_idx, cls in enumerate(class_names):
        paths = sorted(p for p, lab in dataset.samples if lab == cls_idx)
        selected += [(cls, p) for p in paths[:n_per_class]]

    cam = GradCAM(model=model, target_layers=[find_last_conv_layer(model)])

    fig, axes = plt.subplots(len(selected), 2, figsize=(8, 3 * len(selected)))
    if len(selected) == 1:
        axes = axes.reshape(1, -1)

    for i, (cls_name, img_path) in enumerate(selected):
        image = PIL.Image.open(img_path).convert('RGB')
        input_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(input_tensor)
            probs = torch.softmax(logits, dim=1)
            pred_class = logits.argmax(dim=1).item()
            confidence = probs[0, pred_class].item()

        grayscale_cam = cam(input_tensor=input_tensor,
                            targets=[ClassifierOutputTarget(target_class)])[0, :]

        # undo normalization so the overlay sits on a viewable image
        img_denorm = (input_tensor.squeeze(0).cpu()
                      * torch.tensor(std).view(3, 1, 1)
                      + torch.tensor(mean).view(3, 1, 1)).clamp(0, 1)
        rgb_img = img_denorm.permute(1, 2, 0).numpy()

        axes[i, 0].imshow(rgb_img)
        axes[i, 0].set_title(f'actual: {cls_name}')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True))
        axes[i, 1].set_title(f'predicted: {class_names[pred_class]} ({confidence:.0%})')
        axes[i, 1].axis('off')

    fig.suptitle(f'Grad-CAM - {title} (explaining class "{class_names[target_class]}")')
    plt.tight_layout()
    plt.show()

    return grayscale_cam

In [ ]:
def run_tsne(model, data_dir, transform, title='', batch_size=64, device='cpu'):
    """t-SNE of the pooled final-convolution features.

    Hooking the last Conv2d works uniformly across DenseNet, VGG, ResNet and
    EfficientNet, so no per-architecture special-casing is needed.
    """
    model.to(device)
    model.eval()

    features = []
    hook = find_last_conv_layer(model).register_forward_hook(
        lambda module, inp, out: features.append(out.detach().cpu()))

    dataset = ImageFolder(root=data_dir, transform=transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    class_names = dataset.classes

    labels = []
    with torch.no_grad():
        for images, target in loader:
            model(images.to(device))
            labels.extend(target.numpy())

    hook.remove()

    # ReLU then global average pool each feature map to one vector per image
    vectors = torch.cat([F.adaptive_avg_pool2d(F.relu(fm), (1, 1)).flatten(1)
                         for fm in features]).numpy()
    labels = np.array(labels)

    print(f'Collected {vectors.shape[0]} samples with {vectors.shape[1]} features each.')

    # sklearn requires perplexity < n_samples; 30 is the default for real splits
    perplexity = min(30, max(2, len(vectors) - 1))
    embedding = TSNE(n_components=2, random_state=42,
                     perplexity=perplexity).fit_transform(vectors)

    plt.figure(figsize=(8, 6))
    for class_id, colour in zip(range(len(class_names)), ['red', 'blue']):
        mask = labels == class_id
        plt.scatter(embedding[mask, 0], embedding[mask, 1], c=colour,
                    label=class_names[class_id], alpha=0.7, edgecolors='k', s=50)

    plt.title(f't-SNE - {title}')
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    return embedding

# 3. Model Definitions

One builder per architecture, defined once.

In [ ]:
from torchvision.models import (densenet169, DenseNet169_Weights,
                                resnet50, efficientnet_b3)


class CNN(nn.Module):
    """Small custom CNN (Shekar architecture) used as the from-scratch baseline."""

    def __init__(self, n_classes=2):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=128, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.3),

            nn.Conv2d(128, 64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.5),

            nn.Conv2d(64, 64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.5),

            nn.Flatten(),
            nn.Linear(43264, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 2),
        )

    def forward(self, x):
        return self.layers(x)


def build_densenet169():
    """DenseNet169, everything frozen except the norm layers and a new classifier."""
    model = densenet169(weights=DenseNet169_Weights.DEFAULT)
    for name, param in model.named_parameters():
        if 'norm' not in name:
            param.requires_grad = False
    model.classifier = nn.Linear(model.classifier.in_features, 2)
    return model


def build_vgg16():
    """VGG16 with a three-layer classifier head.

    Note: plain VGG16 has no batch-norm layers, so the 'bn' test below freezes
    the whole backbone. Only the new head trains. This matches how the
    manuscript model was trained.
    """
    model = torch.hub.load('pytorch/vision', 'vgg16',
                           weights='VGG16_Weights.IMAGENET1K_V1')
    for name, param in model.named_parameters():
        if 'bn' not in name:
            param.requires_grad = False

    model.classifier = nn.Sequential(
        nn.Linear(model.classifier[0].in_features, 1024),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(1024, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 2),
    )
    return model


def build_resnet50():
    """ResNet50, frozen except batch-norm layers and a new fc head."""
    model = resnet50(weights='DEFAULT')
    for name, param in model.named_parameters():
        if 'bn' not in name:
            param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model


def build_efficientnet_b3():
    """EfficientNetB3, frozen except batch-norm layers and a new classifier head."""
    model = efficientnet_b3(weights='DEFAULT')
    for param in model.parameters():
        param.requires_grad = False
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            for param in module.parameters():
                param.requires_grad = True
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    return model

# 4. Data (D2, 4200 images)

Downloads D2 (Rahman) and builds a stratified **80/10/10** split under
`./data_4200`. The class balance is heavily skewed — 700 TB against 3500 Normal —
which is what the oversampling scheme addresses.

In [ ]:
import kagglehub

path_rahman = kagglehub.dataset_download("tawsifurrahman/tuberculosis-tb-chest-xray-dataset")
print("Path to dataset files:", path_rahman)

In [ ]:
def create_images_folder_4200(path, base_dir=DIR_ORIGINAL):
    """Stratified 80/10/10 split of D2 into base_dir/{train,val,test}/{tb,no_tb}."""

    db = os.path.join(path, 'TB_Chest_Radiography_Database')
    tb_md = pd.read_excel(os.path.join(db, 'Tuberculosis.metadata.xlsx'))
    no_tb_md = pd.read_excel(os.path.join(db, 'Normal.metadata.xlsx'))

    img_id1 = list(tb_md['FILE NAME'])      # TB
    img_id0 = list(no_tb_md['FILE NAME'])   # Normal

    print(f'# of TB images: {len(img_id1)}')
    print(f'# of Normal images: {len(img_id0)}')
    print(f'Total: {len(img_id1) + len(img_id0)}')

    train_ratio, val_ratio = 0.8, 0.1

    # int() floors, so the test split absorbs the remainder
    n_train1, n_train0 = int(train_ratio * len(img_id1)), int(train_ratio * len(img_id0))
    n_val1, n_val0 = int(val_ratio * len(img_id1)), int(val_ratio * len(img_id0))

    splits = {
        'train': (img_id1[:n_train1], img_id0[:n_train0]),
        'val': (img_id1[n_train1:n_train1 + n_val1], img_id0[n_train0:n_train0 + n_val0]),
        'test': (img_id1[n_train1 + n_val1:], img_id0[n_train0 + n_val0:]),
    }

    sources = {'tb': os.path.join(db, 'Tuberculosis'),
               'no_tb': os.path.join(db, 'Normal')}

    for split, (ids_tb, ids_normal) in splits.items():
        for klas, ids in [('tb', ids_tb), ('no_tb', ids_normal)]:
            os.makedirs(os.path.join(base_dir, split, klas), exist_ok=True)
            for i in ids:
                name = f'{i}.png'
                shutil.copy(os.path.join(sources[klas], name),
                            os.path.join(base_dir, split, klas, name))
        print(f'  {split}: {len(ids_tb)} TB + {len(ids_normal)} Normal')

    print(f'Created {base_dir}/train, /val, /test')


create_images_folder_4200(path_rahman)

## Transforms

Declared once here and reused by every model section.

- **ImageNet-style** — DenseNet169, VGG16, ResNet50, EfficientNetB3.
- **Custom-CNN-style** — resize and `ToTensor` only. These D2 checkpoints were
  trained **without normalization**, unlike their D1 counterparts, so applying
  any here would silently invalidate the results.

In [ ]:
# --- ImageNet-style --------------------------------------------------------
TF_IMAGENET_EVAL = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# no-augmentation and oversampling schemes train on the eval transform
TF_IMAGENET_PLAIN = TF_IMAGENET_EVAL

TF_IMAGENET_AUG = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomAffine(degrees=10,            # patients are not always straight
                   translate=(0.1, 0.1),
                   scale=(0.9, 1.1)),     # controls apparent lung size
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# --- Custom CNN: no normalization on D2 ------------------------------------
TF_CNN_EVAL = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

TF_CNN_PLAIN = TF_CNN_EVAL

TF_CNN_AUG = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
])


def make_loaders(base_dir, train_tf, eval_tf, batch_size=32, oversample=False):
    """train/val/test DataLoaders for a base_dir holding train/, val/ and test/.

    oversample=True replaces shuffling with a WeightedRandomSampler that draws
    each class with inverse-frequency probability, so the minority TB class is
    seen as often as Normal.
    """
    train_td = ImageFolder(root=os.path.join(base_dir, 'train'), transform=train_tf)
    val_td = ImageFolder(root=os.path.join(base_dir, 'val'), transform=eval_tf)
    test_td = ImageFolder(root=os.path.join(base_dir, 'test'), transform=eval_tf)

    sampler = None
    if oversample:
        labels = [label for _, label in train_td.samples]
        class_weights = 1.0 / np.bincount(labels)
        sampler = WeightedRandomSampler(
            weights=[class_weights[label] for label in labels],
            num_samples=len(labels), replacement=True)

    train_dl = DataLoader(train_td, batch_size=batch_size,
                          shuffle=(sampler is None), sampler=sampler)
    val_dl = DataLoader(val_td, batch_size=batch_size)
    test_dl = DataLoader(test_td, batch_size=batch_size)

    return train_dl, val_dl, test_dl

# 5. Segmentation Pipeline (for the segmented scheme)

Applies the trained U-Net (`seg_unet_2.pth`) to all 4200 images and multiplies
each by its predicted lung mask — **hard attention**.

Note the ordering: the whole dataset is segmented first, and the ROI images are
then split **70/20/10** from `os.listdir()` order — not the 80/10/10 metadata
split used above. Both leave 10% for test (420 images: 70 TB + 350 Normal), so
the test set is the same size, but not the same images. This is reproduced as-is
because it is how the segmented checkpoints were trained.

In [ ]:
from torchvision.models import vgg16_bn
from torchvision.utils import save_image
from torchvision.transforms import v2 as Tv2


def conv(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


def up_conv(in_channels, out_channels):
    return nn.Sequential(
        nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    """VGG16-BN encoder U-Net. Must match the architecture seg_unet_2.pth was saved from."""

    def __init__(self, num_classes=2):
        super().__init__()

        self.encoder = vgg16_bn(weights='DEFAULT').features

        self.block1 = nn.Sequential(*self.encoder[:6])
        self.block2 = nn.Sequential(*self.encoder[6:13])
        self.block3 = nn.Sequential(*self.encoder[13:20])
        self.block4 = nn.Sequential(*self.encoder[20:27])
        self.block5 = nn.Sequential(*self.encoder[27:34])

        self.bottleneck = nn.Sequential(*self.encoder[34:])
        self.conv_bottleneck = conv(512, 1024)

        self.up_conv6 = up_conv(1024, 512)
        self.conv6 = conv(512 + 512, 512)
        self.up_conv7 = up_conv(512, 256)
        self.conv7 = conv(256 + 512, 256)
        self.up_conv8 = up_conv(256, 128)
        self.conv8 = conv(128 + 256, 128)
        self.up_conv9 = up_conv(128, 64)
        self.conv9 = conv(64 + 128, 64)
        self.up_conv10 = up_conv(64, 32)
        self.conv10 = conv(32 + 64, 32)

        self.conv11 = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        block1 = self.block1(x)
        block2 = self.block2(block1)
        block3 = self.block3(block2)
        block4 = self.block4(block3)
        block5 = self.block5(block4)

        x = self.conv_bottleneck(self.bottleneck(block5))

        x = self.conv6(torch.cat([self.up_conv6(x), block5], dim=1))
        x = self.conv7(torch.cat([self.up_conv7(x), block4], dim=1))
        x = self.conv8(torch.cat([self.up_conv8(x), block3], dim=1))
        x = self.conv9(torch.cat([self.up_conv9(x), block2], dim=1))
        x = self.conv10(torch.cat([self.up_conv10(x), block1], dim=1))

        return self.conv11(x)


class ROI(Dataset):
    """Yields (original, normalized, label, filename) for segmentation inference.

    The normalized copy feeds the U-Net; the un-normalized copy is what the mask
    is applied to, so the saved ROI keeps the original intensities.
    """

    def __init__(self, root_dir):
        self.transforms_norm = Tv2.Compose([
            Tv2.Resize((224, 224)),
            Tv2.ToImage(),
            Tv2.ToDtype(torch.float32, scale=True),
            Tv2.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
        self.transforms_orig = Tv2.Compose([
            Tv2.Resize((224, 224)),
            Tv2.ToImage(),
            Tv2.ToDtype(torch.float32, scale=True),
        ])

        self.root_dir = root_dir
        self.im_data = ([('no_tb', f, 0) for f in os.listdir(os.path.join(root_dir, 'no_tb'))]
                        + [('tb', f, 1) for f in os.listdir(os.path.join(root_dir, 'tb'))])

    def __len__(self):
        return len(self.im_data)

    def __getitem__(self, idx):
        folder, name, label = self.im_data[idx]
        image = PIL.Image.open(os.path.join(self.root_dir, folder, name)).convert('RGB')
        return (self.transforms_orig(image), self.transforms_norm(image),
                torch.tensor(label).long(), name)


def segmentation_inference(model, data_loader, roi_dir, device='cpu'):
    """Saves image * predicted_mask (hard attention) for every image."""
    model.to(device)
    model.eval()

    for klas in ['tb', 'no_tb']:
        os.makedirs(os.path.join(roi_dir, klas), exist_ok=True)

    print(f'Starting inference. Saving images to {roi_dir}...')
    start = time.time()

    with torch.no_grad():
        for batch_idx, (images_orig, images_norm, labels, names) in enumerate(data_loader):
            pred_masks = model(images_norm.to(device)).argmax(dim=1)
            # broadcasting (B,3,H,W) * (B,1,H,W) zeroes everything outside the lungs
            roi_batch = images_orig.to(device) * pred_masks.unsqueeze(1)

            for i in range(len(roi_batch)):
                folder = 'tb' if labels[i].item() == 1 else 'no_tb'
                save_image(roi_batch[i], os.path.join(roi_dir, folder, names[i]),
                           normalize=True)

            if (batch_idx + 1) % 5 == 0:
                print(f'  batch {batch_idx + 1}/{len(data_loader)}  '
                      f'({(time.time() - start) / 60:.2f} min)')

    print(f'Done in {(time.time() - start) / 60:.2f} minutes')

In [ ]:
def create_flat_images_4200(path, base_dir=DIR_FLAT):
    """All 4200 images in base_dir/{tb,no_tb}, unsplit - the U-Net's input."""
    db = os.path.join(path, 'TB_Chest_Radiography_Database')
    tb_md = pd.read_excel(os.path.join(db, 'Tuberculosis.metadata.xlsx'))
    no_tb_md = pd.read_excel(os.path.join(db, 'Normal.metadata.xlsx'))

    for klas, md, source in [('tb', tb_md, os.path.join(db, 'Tuberculosis')),
                             ('no_tb', no_tb_md, os.path.join(db, 'Normal'))]:
        os.makedirs(os.path.join(base_dir, klas), exist_ok=True)
        for i in md['FILE NAME']:
            name = f'{i}.png'
            shutil.copy(os.path.join(source, name),
                        os.path.join(base_dir, klas, name))
        print(f'  {klas}: {len(os.listdir(os.path.join(base_dir, klas)))} images')


def setup_roi_split_4200(roi_dir=DIR_ROI, base_dir=DIR_SEGMENTED):
    """Stratified 70/20/10 split of the ROI images, ordered by os.listdir()."""
    roi_0 = os.listdir(os.path.join(roi_dir, 'no_tb'))
    roi_1 = os.listdir(os.path.join(roi_dir, 'tb'))

    print(f'# of TB images: {len(roi_1)}')
    print(f'# of Normal images: {len(roi_0)}')

    n_train1, n_train0 = int(0.7 * len(roi_1)), int(0.7 * len(roi_0))
    n_val1, n_val0 = int(0.2 * len(roi_1)), int(0.2 * len(roi_0))

    splits = {
        'train': (roi_1[:n_train1], roi_0[:n_train0]),
        'val': (roi_1[n_train1:n_train1 + n_val1], roi_0[n_train0:n_train0 + n_val0]),
        'test': (roi_1[n_train1 + n_val1:], roi_0[n_train0 + n_val0:]),
    }

    for split, (names_tb, names_normal) in splits.items():
        for klas, names in [('tb', names_tb), ('no_tb', names_normal)]:
            os.makedirs(os.path.join(base_dir, split, klas), exist_ok=True)
            for name in names:
                shutil.copy(os.path.join(roi_dir, klas, name),
                            os.path.join(base_dir, split, klas, name))
        print(f'  {split}: {len(names_tb)} TB + {len(names_normal)} Normal')

    print(f'Created {base_dir}/train, /val, /test')


create_flat_images_4200(path_rahman)

unet = UNet(num_classes=2)
unet.load_state_dict(torch.load(SEG_CHECKPOINT, map_location=device))
print(f'Loaded U-Net from {SEG_CHECKPOINT}')

segmentation_inference(unet, DataLoader(ROI(DIR_FLAT), batch_size=64),
                       DIR_ROI, device=device)
setup_roi_split_4200()

## Segmented transforms

ROI images arrive already at 224x224, so no `Resize` is applied. The segmented
Custom CNN was trained with `ToTensor` alone — no resize and no normalization.

In [ ]:
TF_SEG_IMAGENET_TRAIN = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

TF_SEG_IMAGENET_EVAL = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

TF_SEG_CNN_TRAIN = T.Compose([
    T.ToTensor(),
])

TF_SEG_CNN_EVAL = TF_SEG_CNN_TRAIN

# 6. Experiments

Each model gets four subsections, one per scheme. Only the model, the checkpoint
path, the transform and the dataloaders differ — everything else calls the
helpers from section 2.

In [ ]:
# Manuscript checkpoints. VGG16's segmented model lives in the segmentation
# folder rather than the classification one.
def _cls(name):
    return os.path.join(path_classification, name)


CHECKPOINTS = {
    'Custom CNN': {
        'no_aug': _cls('class_ex3_run1_4200.pt'),
        'aug': _cls('class_ex3_run2_4200.pt'),
        'oversampling': _cls('class_ex3_run4_4200.pt'),
        'segmented': _cls('class_seg_3cnn_4200.pt'),
    },
    'DenseNet169': {
        'no_aug': _cls('densenet169_rahman_r2_redo.pt'),
        'aug': _cls('densenet169_rahman_r4.pt'),
        'oversampling': _cls('densenet169_rahman_r5.pt'),
        'segmented': _cls('densenet169_class_seg_rahman.pt'),
    },
    'VGG16': {
        'no_aug': _cls('vgg16_rahman_r1_redo_2.pt'),
        'aug': _cls('vgg16_rahman_r2.pt'),
        'oversampling': _cls('vgg16_rahman_r3_2.pt'),
        'segmented': os.path.join(path_segmentation, 'vgg16_class_seg_4200.pt'),
    },
    'ResNet50': {
        'no_aug': _cls('resnet50_rahman_r3.pt'),
        'aug': _cls('resnet50_rahman_r4.pt'),
        'oversampling': _cls('resnet50_rahman_r5.pt'),
        'segmented': _cls('resnet50_class_seg_4200.pt'),
    },
    'EfficientNetB3': {
        'no_aug': _cls('efficient_net_4200_redo_2.pt'),
        'aug': _cls('efficient_net_4200_run2.pt'),
        'oversampling': _cls('efficient_net_4200_run3.pt'),
        'segmented': _cls('enet_class_seg_4200.pt'),
    },
}

SCHEMES = ['no_aug', 'aug', 'oversampling', 'segmented']

# collects every metric row so the notebook can end with one summary table
results = {}


def record(model_name, scheme, metrics):
    results[(model_name, scheme)] = metrics

## Custom CNN

The three unsegmented checkpoints are **state dicts** and load into a fresh `CNN()`; the segmented one is a full model object. These D2 checkpoints were trained **without normalization**.

### No Augmentation - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_CNN_PLAIN, TF_CNN_EVAL)

model = CNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['Custom CNN']['no_aug'],
                      epochs=100, patience=10,
                      save_mode='state_dict', device=device)
plot_history(history)

### No Augmentation - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_CNN_EVAL, TF_CNN_EVAL)

model = load_checkpoint(CHECKPOINTS['Custom CNN']['no_aug'], how='state_dict', builder=CNN, device=device)
metrics = evaluate_model(model, test_dl, title='Custom CNN (D2, No Augmentation)', device=device)
record('Custom CNN', 'no_aug', metrics)

### No Augmentation - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_ORIGINAL, 'test')

plot_gradcam(model, test_dir, TF_CNN_EVAL, *((0.0, 0.0, 0.0), (1.0, 1.0, 1.0)),
             target_class=1, title='Custom CNN (D2, No Augmentation)', device=device)

_ = run_tsne(model, test_dir, TF_CNN_EVAL,
             title='Custom CNN (D2, No Augmentation)', device=device)

### With Augmentation - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_CNN_AUG, TF_CNN_EVAL)

model = CNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['Custom CNN']['aug'],
                      epochs=100, patience=10,
                      save_mode='state_dict', device=device)
plot_history(history)

### With Augmentation - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_CNN_EVAL, TF_CNN_EVAL)

model = load_checkpoint(CHECKPOINTS['Custom CNN']['aug'], how='state_dict', builder=CNN, device=device)
metrics = evaluate_model(model, test_dl, title='Custom CNN (D2, With Augmentation)', device=device)
record('Custom CNN', 'aug', metrics)

### With Augmentation - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_ORIGINAL, 'test')

plot_gradcam(model, test_dir, TF_CNN_EVAL, *((0.0, 0.0, 0.0), (1.0, 1.0, 1.0)),
             target_class=1, title='Custom CNN (D2, With Augmentation)', device=device)

_ = run_tsne(model, test_dir, TF_CNN_EVAL,
             title='Custom CNN (D2, With Augmentation)', device=device)

### Oversampling - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_CNN_PLAIN, TF_CNN_EVAL, oversample=True)

model = CNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['Custom CNN']['oversampling'],
                      epochs=100, patience=10,
                      save_mode='state_dict', device=device)
plot_history(history)

### Oversampling - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_CNN_EVAL, TF_CNN_EVAL)

model = load_checkpoint(CHECKPOINTS['Custom CNN']['oversampling'], how='state_dict', builder=CNN, device=device)
metrics = evaluate_model(model, test_dl, title='Custom CNN (D2, Oversampling)', device=device)
record('Custom CNN', 'oversampling', metrics)

### Oversampling - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_ORIGINAL, 'test')

plot_gradcam(model, test_dir, TF_CNN_EVAL, *((0.0, 0.0, 0.0), (1.0, 1.0, 1.0)),
             target_class=1, title='Custom CNN (D2, Oversampling)', device=device)

_ = run_tsne(model, test_dir, TF_CNN_EVAL,
             title='Custom CNN (D2, Oversampling)', device=device)

### Segmented - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_SEGMENTED, TF_SEG_CNN_TRAIN, TF_SEG_CNN_EVAL)

model = CNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['Custom CNN']['segmented'],
                      epochs=100, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### Segmented - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_SEGMENTED, TF_SEG_CNN_EVAL, TF_SEG_CNN_EVAL)

model = load_checkpoint(CHECKPOINTS['Custom CNN']['segmented'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='Custom CNN (D2, Segmented)', device=device)
record('Custom CNN', 'segmented', metrics)

### Segmented - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_SEGMENTED, 'test')

plot_gradcam(model, test_dir, TF_SEG_CNN_EVAL, *((0.0, 0.0, 0.0), (1.0, 1.0, 1.0)),
             target_class=1, title='Custom CNN (D2, Segmented)', device=device)

_ = run_tsne(model, test_dir, TF_SEG_CNN_EVAL,
             title='Custom CNN (D2, Segmented)', device=device)

## DenseNet169

### No Augmentation - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_PLAIN, TF_IMAGENET_EVAL)

model = build_densenet169()
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.99))
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['DenseNet169']['no_aug'],
                      epochs=15, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### No Augmentation - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['DenseNet169']['no_aug'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='DenseNet169 (D2, No Augmentation)', device=device)
record('DenseNet169', 'no_aug', metrics)

### No Augmentation - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_ORIGINAL, 'test')

plot_gradcam(model, test_dir, TF_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='DenseNet169 (D2, No Augmentation)', device=device)

_ = run_tsne(model, test_dir, TF_IMAGENET_EVAL,
             title='DenseNet169 (D2, No Augmentation)', device=device)

### With Augmentation - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_AUG, TF_IMAGENET_EVAL)

model = build_densenet169()
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.99))
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['DenseNet169']['aug'],
                      epochs=15, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### With Augmentation - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['DenseNet169']['aug'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='DenseNet169 (D2, With Augmentation)', device=device)
record('DenseNet169', 'aug', metrics)

### With Augmentation - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_ORIGINAL, 'test')

plot_gradcam(model, test_dir, TF_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='DenseNet169 (D2, With Augmentation)', device=device)

_ = run_tsne(model, test_dir, TF_IMAGENET_EVAL,
             title='DenseNet169 (D2, With Augmentation)', device=device)

### Oversampling - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_PLAIN, TF_IMAGENET_EVAL, oversample=True)

model = build_densenet169()
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.99))
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['DenseNet169']['oversampling'],
                      epochs=15, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### Oversampling - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['DenseNet169']['oversampling'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='DenseNet169 (D2, Oversampling)', device=device)
record('DenseNet169', 'oversampling', metrics)

### Oversampling - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_ORIGINAL, 'test')

plot_gradcam(model, test_dir, TF_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='DenseNet169 (D2, Oversampling)', device=device)

_ = run_tsne(model, test_dir, TF_IMAGENET_EVAL,
             title='DenseNet169 (D2, Oversampling)', device=device)

### Segmented - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_SEGMENTED, TF_SEG_IMAGENET_TRAIN, TF_SEG_IMAGENET_EVAL)

model = build_densenet169()
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.99))
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['DenseNet169']['segmented'],
                      epochs=15, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### Segmented - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_SEGMENTED, TF_SEG_IMAGENET_EVAL, TF_SEG_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['DenseNet169']['segmented'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='DenseNet169 (D2, Segmented)', device=device)
record('DenseNet169', 'segmented', metrics)

### Segmented - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_SEGMENTED, 'test')

plot_gradcam(model, test_dir, TF_SEG_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='DenseNet169 (D2, Segmented)', device=device)

_ = run_tsne(model, test_dir, TF_SEG_IMAGENET_EVAL,
             title='DenseNet169 (D2, Segmented)', device=device)

## VGG16

### No Augmentation - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_PLAIN, TF_IMAGENET_EVAL)

model = build_vgg16()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['VGG16']['no_aug'],
                      epochs=20, patience=10,
                      save_mode='full', device=device)
plot_history(history)

### No Augmentation - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['VGG16']['no_aug'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='VGG16 (D2, No Augmentation)', device=device)
record('VGG16', 'no_aug', metrics)

### No Augmentation - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_ORIGINAL, 'test')

plot_gradcam(model, test_dir, TF_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='VGG16 (D2, No Augmentation)', device=device)

_ = run_tsne(model, test_dir, TF_IMAGENET_EVAL,
             title='VGG16 (D2, No Augmentation)', device=device)

### With Augmentation - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_AUG, TF_IMAGENET_EVAL)

model = build_vgg16()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['VGG16']['aug'],
                      epochs=20, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### With Augmentation - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['VGG16']['aug'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='VGG16 (D2, With Augmentation)', device=device)
record('VGG16', 'aug', metrics)

### With Augmentation - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_ORIGINAL, 'test')

plot_gradcam(model, test_dir, TF_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='VGG16 (D2, With Augmentation)', device=device)

_ = run_tsne(model, test_dir, TF_IMAGENET_EVAL,
             title='VGG16 (D2, With Augmentation)', device=device)

### Oversampling - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_PLAIN, TF_IMAGENET_EVAL, oversample=True)

model = build_vgg16()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['VGG16']['oversampling'],
                      epochs=20, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### Oversampling - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['VGG16']['oversampling'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='VGG16 (D2, Oversampling)', device=device)
record('VGG16', 'oversampling', metrics)

### Oversampling - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_ORIGINAL, 'test')

plot_gradcam(model, test_dir, TF_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='VGG16 (D2, Oversampling)', device=device)

_ = run_tsne(model, test_dir, TF_IMAGENET_EVAL,
             title='VGG16 (D2, Oversampling)', device=device)

### Segmented - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_SEGMENTED, TF_SEG_IMAGENET_TRAIN, TF_SEG_IMAGENET_EVAL)

model = build_vgg16()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['VGG16']['segmented'],
                      epochs=20, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### Segmented - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_SEGMENTED, TF_SEG_IMAGENET_EVAL, TF_SEG_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['VGG16']['segmented'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='VGG16 (D2, Segmented)', device=device)
record('VGG16', 'segmented', metrics)

### Segmented - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_SEGMENTED, 'test')

plot_gradcam(model, test_dir, TF_SEG_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='VGG16 (D2, Segmented)', device=device)

_ = run_tsne(model, test_dir, TF_SEG_IMAGENET_EVAL,
             title='VGG16 (D2, Segmented)', device=device)

## ResNet50

### No Augmentation - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_PLAIN, TF_IMAGENET_EVAL)

model = build_resnet50()
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.99))
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['ResNet50']['no_aug'],
                      epochs=15, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### No Augmentation - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['ResNet50']['no_aug'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='ResNet50 (D2, No Augmentation)', device=device)
record('ResNet50', 'no_aug', metrics)

### No Augmentation - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_ORIGINAL, 'test')

plot_gradcam(model, test_dir, TF_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='ResNet50 (D2, No Augmentation)', device=device)

_ = run_tsne(model, test_dir, TF_IMAGENET_EVAL,
             title='ResNet50 (D2, No Augmentation)', device=device)

### With Augmentation - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_AUG, TF_IMAGENET_EVAL)

model = build_resnet50()
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.99))
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['ResNet50']['aug'],
                      epochs=15, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### With Augmentation - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['ResNet50']['aug'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='ResNet50 (D2, With Augmentation)', device=device)
record('ResNet50', 'aug', metrics)

### With Augmentation - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_ORIGINAL, 'test')

plot_gradcam(model, test_dir, TF_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='ResNet50 (D2, With Augmentation)', device=device)

_ = run_tsne(model, test_dir, TF_IMAGENET_EVAL,
             title='ResNet50 (D2, With Augmentation)', device=device)

### Oversampling - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_PLAIN, TF_IMAGENET_EVAL, oversample=True)

model = build_resnet50()
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.99))
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['ResNet50']['oversampling'],
                      epochs=15, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### Oversampling - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['ResNet50']['oversampling'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='ResNet50 (D2, Oversampling)', device=device)
record('ResNet50', 'oversampling', metrics)

### Oversampling - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_ORIGINAL, 'test')

plot_gradcam(model, test_dir, TF_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='ResNet50 (D2, Oversampling)', device=device)

_ = run_tsne(model, test_dir, TF_IMAGENET_EVAL,
             title='ResNet50 (D2, Oversampling)', device=device)

### Segmented - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_SEGMENTED, TF_SEG_IMAGENET_TRAIN, TF_SEG_IMAGENET_EVAL)

model = build_resnet50()
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.99))
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['ResNet50']['segmented'],
                      epochs=20, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### Segmented - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_SEGMENTED, TF_SEG_IMAGENET_EVAL, TF_SEG_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['ResNet50']['segmented'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='ResNet50 (D2, Segmented)', device=device)
record('ResNet50', 'segmented', metrics)

### Segmented - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_SEGMENTED, 'test')

plot_gradcam(model, test_dir, TF_SEG_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='ResNet50 (D2, Segmented)', device=device)

_ = run_tsne(model, test_dir, TF_SEG_IMAGENET_EVAL,
             title='ResNet50 (D2, Segmented)', device=device)

## EfficientNetB3

### No Augmentation - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_PLAIN, TF_IMAGENET_EVAL)

model = build_efficientnet_b3()
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.99))
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['EfficientNetB3']['no_aug'],
                      epochs=20, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### No Augmentation - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['EfficientNetB3']['no_aug'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='EfficientNetB3 (D2, No Augmentation)', device=device)
record('EfficientNetB3', 'no_aug', metrics)

### No Augmentation - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_ORIGINAL, 'test')

plot_gradcam(model, test_dir, TF_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='EfficientNetB3 (D2, No Augmentation)', device=device)

_ = run_tsne(model, test_dir, TF_IMAGENET_EVAL,
             title='EfficientNetB3 (D2, No Augmentation)', device=device)

### With Augmentation - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_AUG, TF_IMAGENET_EVAL)

model = build_efficientnet_b3()
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.99))
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['EfficientNetB3']['aug'],
                      epochs=20, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### With Augmentation - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['EfficientNetB3']['aug'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='EfficientNetB3 (D2, With Augmentation)', device=device)
record('EfficientNetB3', 'aug', metrics)

### With Augmentation - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_ORIGINAL, 'test')

plot_gradcam(model, test_dir, TF_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='EfficientNetB3 (D2, With Augmentation)', device=device)

_ = run_tsne(model, test_dir, TF_IMAGENET_EVAL,
             title='EfficientNetB3 (D2, With Augmentation)', device=device)

### Oversampling - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_PLAIN, TF_IMAGENET_EVAL, oversample=True)

model = build_efficientnet_b3()
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.99))
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['EfficientNetB3']['oversampling'],
                      epochs=20, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### Oversampling - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['EfficientNetB3']['oversampling'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='EfficientNetB3 (D2, Oversampling)', device=device)
record('EfficientNetB3', 'oversampling', metrics)

### Oversampling - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_ORIGINAL, 'test')

plot_gradcam(model, test_dir, TF_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='EfficientNetB3 (D2, Oversampling)', device=device)

_ = run_tsne(model, test_dir, TF_IMAGENET_EVAL,
             title='EfficientNetB3 (D2, Oversampling)', device=device)

### Segmented - Training

In [ ]:
train_dl, val_dl, test_dl = make_loaders(DIR_SEGMENTED, TF_SEG_IMAGENET_TRAIN, TF_SEG_IMAGENET_EVAL)

model = build_efficientnet_b3()
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.99))
loss_fn = nn.CrossEntropyLoss()

history = train_model(model, train_dl, val_dl, optimizer, loss_fn,
                      save_path=CHECKPOINTS['EfficientNetB3']['segmented'],
                      epochs=20, patience=5,
                      save_mode='full', device=device)
plot_history(history)

### Segmented - Evaluation

In [ ]:
_, _, test_dl = make_loaders(DIR_SEGMENTED, TF_SEG_IMAGENET_EVAL, TF_SEG_IMAGENET_EVAL)

model = load_checkpoint(CHECKPOINTS['EfficientNetB3']['segmented'], how='full', device=device)
metrics = evaluate_model(model, test_dl, title='EfficientNetB3 (D2, Segmented)', device=device)
record('EfficientNetB3', 'segmented', metrics)

### Segmented - Grad-CAM and t-SNE

In [ ]:
test_dir = os.path.join(DIR_SEGMENTED, 'test')

plot_gradcam(model, test_dir, TF_SEG_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
             target_class=1, title='EfficientNetB3 (D2, Segmented)', device=device)

_ = run_tsne(model, test_dir, TF_SEG_IMAGENET_EVAL,
             title='EfficientNetB3 (D2, Segmented)', device=device)

## ViT-B/16

Vision Transformer baseline against the five CNNs above. This is an **exploratory
pass** - the aim is a working, reasonably tuned setup to see whether ViT is viable
at this dataset size, not a final polished result.

**Scope: two schemes x two fine-tuning strategies = four runs.** Oversampling and
Segmented are deliberately *not* run for ViT yet; weighted-sampling repetition
interacts differently with attention-based models, so a stable baseline comes first.

| Run key | Scheme | Backbone | LR | Checkpoint |
|---|---|---|---|---|
| `no_aug_frozen` | No Augmentation | frozen, head only | 1e-3 | `vit_b16_d2_no_aug_frozen.pt` |
| `no_aug_full_ft` | No Augmentation | full fine-tune | 1e-4 | `vit_b16_d2_no_aug_full_ft.pt` |
| `aug_frozen` | With Augmentation | frozen, head only | 1e-3 | `vit_b16_d2_aug_frozen.pt` |
| `aug_full_ft` | With Augmentation | full fine-tune | 1e-4 | `vit_b16_d2_aug_full_ft.pt` |

**Reused unchanged from sections 2-4:** the `TF_IMAGENET_*` transforms (ViT takes
the same ImageNet normalization and 224x224 input as the pretrained CNNs, so no
ViT-specific transform is needed), `make_loaders` and therefore the same 80/10/10
split as every other model, `evaluate_model`, `plot_history` and `load_checkpoint`.

**ViT-specific, defined below:** the model builder, an AdamW + warmup/cosine
schedule, a mixed-precision training loop whose early stopping is held off until
warmup finishes, and Grad-CAM / t-SNE helpers that work on tokens rather than a 2D
convolutional feature map. Section 2's CNN helpers are left untouched - later ViT
experiments reuse the helpers defined here.

Checkpoints are saved as **state dicts**, so they reload with
`load_checkpoint(..., how='state_dict', builder=...)` rather than the CNN sections'
full-object convention.

The four Evaluation cells report **validation** metrics - that is what the
frozen-vs-full-fine-tune decision is made on. Test-set evaluation is held back to a
single cell at the end of the section, run once for the winning configuration.

### ViT helpers

In [ ]:
from torchvision.models import vit_b_16, ViT_B_16_Weights


def build_vit_b16(freeze_backbone=False, n_classes=2):
    """ViT-B/16 with ImageNet-1k weights and a fresh binary head.

    Written separately from the CNN builders in section 3 rather than reusing
    their structure: torchvision's VisionTransformer exposes its classifier as
    `heads.head`, and the switch compared here is backbone-level rather than the
    per-parameter-name ('bn' / 'norm') freezing the CNN builders do.

      freeze_backbone=True   encoder frozen, only the new head trains  (LR 1e-3)
      freeze_backbone=False  everything trains end to end              (LR 1e-4)

    No dropout is inserted by hand: every ViT encoder block already contains
    dropout, left at its default rate for this baseline. Raising it is a later
    tuning lever, only if overfitting shows up.
    """
    model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # replaced after freezing, so the new head is always trainable
    model.heads.head = nn.Linear(model.heads.head.in_features, n_classes)
    return model


# --- structure check -------------------------------------------------------
# Grad-CAM below targets `encoder.layers[-1].ln_1`. Confirm the attribute names
# on this torchvision build before anything depends on them.
_vit = build_vit_b16()
print('heads children      :', list(dict(_vit.heads.named_children())))
print('encoder children    :', list(dict(_vit.encoder.named_children())))
print('last block children :', list(dict(_vit.encoder.layers[-1].named_children())))
print('classifier head     :', _vit.heads.head)
print('Grad-CAM target     :', _vit.encoder.layers[-1].ln_1)
del _vit

In [ ]:
import math
from torch.optim.lr_scheduler import LambdaLR


def make_vit_optimizer_and_scheduler(model, train_dl, epochs, lr,
                                     weight_decay=0.05, warmup_frac=0.05):
    """AdamW with a linear warmup into a cosine decay, stepped once per batch.

    Intentionally different from the CNN sections, which all use plain
    `optim.Adam(lr=0.001)` with no schedule at all:

      * AdamW decouples weight decay from the gradient update, which is what the
        0.05 decay used by ViT recipes assumes. Plain Adam's L2-coupled decay
        behaves quite differently at that magnitude.
      * The warmup matters far more here than it did for the CNNs. Attention
        blocks are unstable in the first few hundred steps at full LR, and
        skipping warmup is a common cause of a ViT fine-tune diverging or
        collapsing onto a single class. Do not drop it.

    Only parameters with requires_grad are handed to the optimizer, so the
    frozen-backbone run optimizes the head alone.

    Returns (optimizer, scheduler, warmup_epochs); warmup_epochs is what the
    training loop uses to hold off early stopping while the LR is still ramping.
    """
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(params, lr=lr, weight_decay=weight_decay)

    steps_per_epoch = len(train_dl)
    total_steps = steps_per_epoch * epochs
    warmup_steps = max(1, int(warmup_frac * total_steps))

    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    scheduler = LambdaLR(optimizer, lr_lambda)
    warmup_epochs = math.ceil(warmup_steps / steps_per_epoch)

    print(f'{sum(p.numel() for p in params):,} trainable params  |  lr {lr:g}  '
          f'wd {weight_decay}  |  {total_steps} steps, {warmup_steps} of them '
          f'warmup ({warmup_epochs} epoch(s))')
    return optimizer, scheduler, warmup_epochs

In [ ]:
def train_vit(model, train_dl, val_dl, optimizer, scheduler, loss_fn, save_path,
              epochs=20, patience=10, warmup_epochs=1, device='cpu'):
    """ViT training loop: mixed precision, per-batch LR schedule, warmup-gated
    early stopping, best-validation-loss checkpointing.

    Same contract as section 2's `train_model` - it returns the identical history
    dict, so `plot_history(history)` works on it unchanged - but it cannot simply
    be that function, because ViT needs three things the CNN loop has no notion of:

      1. Mixed precision (torch.amp, the current spelling of the torch.cuda.amp
         path). ViT-B/16 is much slower per epoch than these CNNs and this runs
         on a free-tier Colab GPU.
      2. A scheduler stepped every *batch*, since warmup and cosine decay are
         defined over steps.
      3. Early stopping that does not count patience during warmup. Val loss is
         unreliable while the LR is still ramping up, so a poor warmup epoch must
         not burn patience or stop the run prematurely.

    Checkpointing runs from epoch 1 - only the *stopping* is gated. The best
    checkpoint is a state dict, so reload it with
    `load_checkpoint(path, how='state_dict', builder=..., device=device)`.
    """
    model.to(device)

    use_amp = torch.device(device).type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    history = {'Training': {'Loss': [], 'Accuracy': [], 'Time': []},
               'Validation': {'Loss': [], 'Accuracy': [], 'Time': []}}

    best_val_loss = np.inf
    count = 0

    for epoch in range(1, epochs + 1):

        # --- train ---
        model.train()
        t0 = time.time()
        train_loss = 0
        correct = 0

        for batch, target in train_dl:
            inputs, target = batch.to(device), target.to(device)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                output = model(inputs)
                loss = loss_fn(output, target)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            train_loss += loss.item()
            correct += (output.argmax(dim=1) == target).sum().item()

        history['Training']['Loss'].append(train_loss / len(train_dl))
        history['Training']['Accuracy'].append(correct / len(train_dl.dataset))
        history['Training']['Time'].append((time.time() - t0) / 60)

        # --- validate ---
        model.eval()
        t0 = time.time()
        val_loss = 0
        correct = 0

        with torch.no_grad():
            for batch, target in val_dl:
                inputs, target = batch.to(device), target.to(device)
                with torch.amp.autocast('cuda', enabled=use_amp):
                    output = model(inputs)
                    val_loss += loss_fn(output, target).item()
                correct += (output.argmax(dim=1) == target).sum().item()

        val_loss /= len(val_dl)
        val_acc = correct / len(val_dl.dataset)

        history['Validation']['Loss'].append(val_loss)
        history['Validation']['Accuracy'].append(val_acc)
        history['Validation']['Time'].append((time.time() - t0) / 60)

        # --- checkpoint (always) + early stopping (only once warmup is done) ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            count = 0
            torch.save(model.state_dict(), save_path)
            print(f'Best model saved at epoch {epoch}')
        elif epoch > warmup_epochs:
            count += 1

        print(f"Epoch [{epoch}/{epochs}]  "
              f"lr {optimizer.param_groups[0]['lr']:.2e}  "
              f"train loss {history['Training']['Loss'][-1]:.4f}  "
              f"acc {history['Training']['Accuracy'][-1]:.4f}  |  "
              f"val loss {val_loss:.4f}  acc {val_acc:.4f}"
              + ('   [warmup - early stopping held off]'
                 if epoch <= warmup_epochs else f'   [patience {count}/{patience}]'))

        if epoch > warmup_epochs and count >= patience:
            print(f'Early stopping triggered at epoch {epoch}')
            break

    print(f'\nBest validation loss: {best_val_loss:.4f}')
    print(f"Total training time: {sum(history['Training']['Time']):.2f} min")
    return history

In [ ]:
def vit_reshape_transform(tensor, height=14, width=14):
    """Token sequence -> 2D feature map, for Grad-CAM.

    An encoder block emits (B, 1 + 196, 768) for a 224x224 input at patch size
    16. Drop the [CLS] token at position 0 and fold the 196 patch tokens back
    into the 14x14 grid they came from, so the CAM machinery sees something
    shaped like a convolutional feature map.
    """
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    return result.permute(0, 3, 1, 2)


def plot_gradcam_vit(model, data_dir, transform, mean, std, target_class=1,
                     n_per_class=2, title='', device='cpu'):
    """Grad-CAM overlays for ViT. The overlay, denormalization and plotting code
    is identical to section 2's `plot_gradcam`; only the CAM extraction differs.

    ViT has no 2D convolutional feature map, so `find_last_conv_layer` is the
    wrong target here - the network's only Conv2d is the patch-embedding
    `conv_proj`, whose output has seen no attention at all. Target the first
    LayerNorm of the last encoder block instead, and reshape its tokens to a grid.
    """
    model.to(device)
    # gradients must flow even though the backbone was frozen for training
    for param in model.parameters():
        param.requires_grad = True
    model.eval()

    dataset = ImageFolder(root=data_dir, transform=None)
    class_names = dataset.classes

    selected = []
    for cls_idx, cls in enumerate(class_names):
        paths = sorted(p for p, lab in dataset.samples if lab == cls_idx)
        selected += [(cls, p) for p in paths[:n_per_class]]

    cam = GradCAM(model=model,
                  target_layers=[model.encoder.layers[-1].ln_1],
                  reshape_transform=vit_reshape_transform)

    fig, axes = plt.subplots(len(selected), 2, figsize=(8, 3 * len(selected)))
    if len(selected) == 1:
        axes = axes.reshape(1, -1)

    for i, (cls_name, img_path) in enumerate(selected):
        image = PIL.Image.open(img_path).convert('RGB')
        input_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(input_tensor)
            probs = torch.softmax(logits, dim=1)
            pred_class = logits.argmax(dim=1).item()
            confidence = probs[0, pred_class].item()

        grayscale_cam = cam(input_tensor=input_tensor,
                            targets=[ClassifierOutputTarget(target_class)])[0, :]

        # undo normalization so the overlay sits on a viewable image
        img_denorm = (input_tensor.squeeze(0).cpu()
                      * torch.tensor(std).view(3, 1, 1)
                      + torch.tensor(mean).view(3, 1, 1)).clamp(0, 1)
        rgb_img = img_denorm.permute(1, 2, 0).numpy()

        axes[i, 0].imshow(rgb_img)
        axes[i, 0].set_title(f'actual: {cls_name}')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True))
        axes[i, 1].set_title(f'predicted: {class_names[pred_class]} ({confidence:.0%})')
        axes[i, 1].axis('off')

    fig.suptitle(f'Grad-CAM - {title} (explaining class "{class_names[target_class]}")')
    plt.tight_layout()
    plt.show()

    return grayscale_cam


def run_tsne_vit(model, data_dir, transform, title='', batch_size=64, device='cpu'):
    """t-SNE of the pre-head [CLS] representation.

    Section 2's `run_tsne` hooks the last Conv2d and average-pools the feature
    map; ViT has neither. torchvision's forward slices the [CLS] token out of the
    encoder output and hands it straight to `heads`, so a forward *pre* hook on
    `heads` captures exactly the 768-d vector the classifier sees - already one
    vector per image, so no ReLU/pooling step is needed. Everything downstream of
    feature extraction is the same as the CNN version.
    """
    model.to(device)
    model.eval()

    features = []
    hook = model.heads.register_forward_pre_hook(
        lambda module, inp: features.append(inp[0].detach().cpu()))

    dataset = ImageFolder(root=data_dir, transform=transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    class_names = dataset.classes

    labels = []
    with torch.no_grad():
        for images, target in loader:
            model(images.to(device))
            labels.extend(target.numpy())

    hook.remove()

    vectors = torch.cat(features).numpy()
    labels = np.array(labels)

    print(f'Collected {vectors.shape[0]} samples with {vectors.shape[1]} features each.')

    # sklearn requires perplexity < n_samples; 30 is the default for real splits
    perplexity = min(30, max(2, len(vectors) - 1))
    embedding = TSNE(n_components=2, random_state=42,
                     perplexity=perplexity).fit_transform(vectors)

    plt.figure(figsize=(8, 6))
    for class_id, colour in zip(range(len(class_names)), ['red', 'blue']):
        mask = labels == class_id
        plt.scatter(embedding[mask, 0], embedding[mask, 1], c=colour,
                    label=class_names[class_id], alpha=0.7, edgecolors='k', s=50)

    plt.title(f't-SNE - {title}')
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    return embedding

In [ ]:
# ViT checkpoints, kept separate from CHECKPOINTS so section 7 is unaffected.
# Saved as state dicts, so they reload through a builder rather than as objects.
VIT_CHECKPOINTS = {
    'no_aug_frozen':  _cls('vit_b16_d2_no_aug_frozen.pt'),
    'no_aug_full_ft': _cls('vit_b16_d2_no_aug_full_ft.pt'),
    'aug_frozen':     _cls('vit_b16_d2_aug_frozen.pt'),
    'aug_full_ft':    _cls('vit_b16_d2_aug_full_ft.pt'),
}

# validation metrics per run, so the four are easy to line up afterwards
vit_results = {}


def record_vit(run_key, metrics):
    vit_results[run_key] = metrics

### No Augmentation, frozen backbone - Training

In [ ]:
# ViT-B/16 (D2, No Augmentation, frozen backbone)
# LR 1e-3 suits head-only training, where the pretrained features are
# fixed and only a linear layer has to move. A starting point, not a
# validated optimum - expect to tune it.
EPOCHS, PATIENCE, LR = 30, 10, 1e-3

train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_PLAIN, TF_IMAGENET_EVAL)

model = build_vit_b16(freeze_backbone=True)
optimizer, scheduler, warmup_epochs = make_vit_optimizer_and_scheduler(
    model, train_dl, epochs=EPOCHS, lr=LR)
loss_fn = nn.CrossEntropyLoss()

history = train_vit(model, train_dl, val_dl, optimizer, scheduler, loss_fn,
                    save_path=VIT_CHECKPOINTS['no_aug_frozen'],
                    epochs=EPOCHS, patience=PATIENCE,
                    warmup_epochs=warmup_epochs, device=device)
plot_history(history)

### No Augmentation, frozen backbone - Validation

In [ ]:
_, val_dl, _ = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(VIT_CHECKPOINTS['no_aug_frozen'], how='state_dict',
                        builder=lambda: build_vit_b16(freeze_backbone=True),
                        device=device)
metrics = evaluate_model(model, val_dl, device=device,
                         title='ViT-B/16 (D2, No Augmentation, frozen backbone) - validation')
record_vit('no_aug_frozen', metrics)

### No Augmentation, full fine-tuning - Training

In [ ]:
# ViT-B/16 (D2, No Augmentation, full fine-tuning)
# LR 1e-4 for full fine-tuning: an order of magnitude below the frozen
# run, because the pretrained attention weights are now moving too. A
# starting point, not a validated optimum - expect to tune it.
EPOCHS, PATIENCE, LR = 20, 10, 1e-4

train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_PLAIN, TF_IMAGENET_EVAL)

model = build_vit_b16(freeze_backbone=False)
optimizer, scheduler, warmup_epochs = make_vit_optimizer_and_scheduler(
    model, train_dl, epochs=EPOCHS, lr=LR)
loss_fn = nn.CrossEntropyLoss()

history = train_vit(model, train_dl, val_dl, optimizer, scheduler, loss_fn,
                    save_path=VIT_CHECKPOINTS['no_aug_full_ft'],
                    epochs=EPOCHS, patience=PATIENCE,
                    warmup_epochs=warmup_epochs, device=device)
plot_history(history)

### No Augmentation, full fine-tuning - Validation

In [ ]:
_, val_dl, _ = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(VIT_CHECKPOINTS['no_aug_full_ft'], how='state_dict',
                        builder=lambda: build_vit_b16(freeze_backbone=False),
                        device=device)
metrics = evaluate_model(model, val_dl, device=device,
                         title='ViT-B/16 (D2, No Augmentation, full fine-tuning) - validation')
record_vit('no_aug_full_ft', metrics)

### No Augmentation - Grad-CAM and t-SNE

Run once, on a no-augmentation model, purely to confirm the ViT explainability
path works end to end - the late-block LayerNorm target plus the token-to-grid
reshape are the only things that differ from the CNN version, and they are the
parts that can silently produce a garbage map. Once the four runs have been
compared on validation, figures only need generating for the winner.

In [ ]:
GRADCAM_RUN = 'no_aug_full_ft'      # switch to 'no_aug_frozen' to check that one

test_dir = os.path.join(DIR_ORIGINAL, 'test')

model = load_checkpoint(VIT_CHECKPOINTS[GRADCAM_RUN], how='state_dict',
                        builder=lambda: build_vit_b16(),
                        device=device)

plot_gradcam_vit(model, test_dir, TF_IMAGENET_EVAL, *(IMAGENET_MEAN, IMAGENET_STD),
                 target_class=1, title=f'ViT-B/16 (D2, {GRADCAM_RUN})', device=device)

_ = run_tsne_vit(model, test_dir, TF_IMAGENET_EVAL,
                 title=f'ViT-B/16 (D2, {GRADCAM_RUN})', device=device)

### With Augmentation, frozen backbone - Training

In [ ]:
# ViT-B/16 (D2, With Augmentation, frozen backbone)
# Same settings as the no-aug frozen run; only the train transform changes.
EPOCHS, PATIENCE, LR = 30, 10, 1e-3

train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_AUG, TF_IMAGENET_EVAL)

model = build_vit_b16(freeze_backbone=True)
optimizer, scheduler, warmup_epochs = make_vit_optimizer_and_scheduler(
    model, train_dl, epochs=EPOCHS, lr=LR)
loss_fn = nn.CrossEntropyLoss()

history = train_vit(model, train_dl, val_dl, optimizer, scheduler, loss_fn,
                    save_path=VIT_CHECKPOINTS['aug_frozen'],
                    epochs=EPOCHS, patience=PATIENCE,
                    warmup_epochs=warmup_epochs, device=device)
plot_history(history)

### With Augmentation, frozen backbone - Validation

In [ ]:
_, val_dl, _ = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(VIT_CHECKPOINTS['aug_frozen'], how='state_dict',
                        builder=lambda: build_vit_b16(freeze_backbone=True),
                        device=device)
metrics = evaluate_model(model, val_dl, device=device,
                         title='ViT-B/16 (D2, With Augmentation, frozen backbone) - validation')
record_vit('aug_frozen', metrics)

### With Augmentation, full fine-tuning - Training

In [ ]:
# ViT-B/16 (D2, With Augmentation, full fine-tuning)
# Same settings as the no-aug full fine-tune; only the train transform changes.
EPOCHS, PATIENCE, LR = 20, 10, 1e-4

train_dl, val_dl, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_AUG, TF_IMAGENET_EVAL)

model = build_vit_b16(freeze_backbone=False)
optimizer, scheduler, warmup_epochs = make_vit_optimizer_and_scheduler(
    model, train_dl, epochs=EPOCHS, lr=LR)
loss_fn = nn.CrossEntropyLoss()

history = train_vit(model, train_dl, val_dl, optimizer, scheduler, loss_fn,
                    save_path=VIT_CHECKPOINTS['aug_full_ft'],
                    epochs=EPOCHS, patience=PATIENCE,
                    warmup_epochs=warmup_epochs, device=device)
plot_history(history)

### With Augmentation, full fine-tuning - Validation

In [ ]:
_, val_dl, _ = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(VIT_CHECKPOINTS['aug_full_ft'], how='state_dict',
                        builder=lambda: build_vit_b16(freeze_backbone=False),
                        device=device)
metrics = evaluate_model(model, val_dl, device=device,
                         title='ViT-B/16 (D2, With Augmentation, full fine-tuning) - validation')
record_vit('aug_full_ft', metrics)

### Test set - best configuration only

Held back on purpose. The four runs above are compared on **validation**, so the
D2 test split is not read four times over during model selection. Set `BEST_RUN`
to whichever configuration won and run this once.

In [ ]:
BEST_RUN = 'no_aug_full_ft'      # <- set from the validation results above

_, _, test_dl = make_loaders(DIR_ORIGINAL, TF_IMAGENET_EVAL, TF_IMAGENET_EVAL)

model = load_checkpoint(VIT_CHECKPOINTS[BEST_RUN], how='state_dict',
                        builder=lambda: build_vit_b16(),
                        device=device)
_ = evaluate_model(model, test_dl, title=f'ViT-B/16 (D2, {BEST_RUN}) - test',
                   device=device)

# 7. Summary Table

In [ ]:
summary = pd.DataFrame([
    {'Model': model_name, 'Scheme': scheme, **metrics}
    for (model_name, scheme), metrics in results.items()
]).set_index(['Model', 'Scheme'])

# configuration order rather than execution order
order = [(m, s) for m in CHECKPOINTS for s in SCHEMES]
summary = summary.reindex([k for k in order if k in summary.index])

print('D2 (4200) results on the D2 test split')
display(summary.round(4))